In [9]:
import joblib
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ══════════════════════════════════════════════════════════════
# Reproducibility
# Ensures the same random initialization and data shuffling
# every time the script is run.
# ══════════════════════════════════════════════════════════════
np.random.seed(42)
torch.manual_seed(42)


# ══════════════════════════════════════════════════════════════
# STEP 1: LOAD THE DATA
# We load the BETH dataset which contains system logs.
# Each row represents a single event on the network.
# ══════════════════════════════════════════════════════════════

import os

csv_path = "data/labelled_training_data.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"Dataset '{csv_path}' was not found. "
        "Please check the file path before training."
    )

df = pd.read_csv(csv_path)


# These are the columns we use to detect threats
# Think of them as "clues" the model learns from
features = [
    'userId',           # Who is logged in
    'mountNamespace',   # What filesystem restrictions apply
    'argsNum',          # How many arguments were passed
    'returnValue'       # What the event returned (usually 0)
]

X = df[features].values   # Input features (the clues)
y = df['sus_label'].values # Target: 1 = Malicious, 0 = Benign


# ══════════════════════════════════════════════════════════════
# STEP 2: PREPARE THE DATA
# Split into Train (70%), Validation (15%), and Test (15%).
# This prevents overfitting to the validation set and keeps
# the test set completely unseen until the end.
# ══════════════════════════════════════════════════════════════

# First split: 70% Train, 30% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: 15% Validation, 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Compute how much rarer malicious samples are
num_negative = np.sum(y_train == 0)
num_positive = np.sum(y_train == 1)

pos_weight = torch.tensor(
    [num_negative / num_positive],
    dtype=torch.float32
)

# Prevent data leakage:
# Learn scaling parameters ONLY from the training data.
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t   = torch.tensor(y_val, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=64,
    shuffle=True
)

# ══════════════════════════════════════════════════════════════
# STEP 3: BUILD THE NEURAL NETWORK
# This is the brain of our detector.
# It has 3 layers that progressively extract patterns:
#   Layer 1: 7 inputs  → 64 neurons (broad pattern detection)
#   Layer 2: 64 neurons → 32 neurons (refine patterns)
#   Layer 3: 32 neurons →  1 output  (malicious or benign?)
#
# Dropout randomly switches off 30% of neurons during training
# to prevent the model from over-relying on any single feature.
#
# Sigmoid squashes the final output to a value between 0 and 1
# which we interpret as a probability of being malicious.
# ══════════════════════════════════════════════════════════════
class CyberThreatDetector(nn.Module):
    def __init__(self, input_dim):
        super(CyberThreatDetector, self).__init__()
        self.network = nn.Sequential(

            nn.Linear(input_dim, 64), # Layer 1: expand and detect
            nn.ReLU(),                # Activation: ignore negatives
            nn.Dropout(0.3),          # Regularization: prevent overfitting

            nn.Linear(64, 32),        # Layer 2: compress and refine
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 1),         # Layer 3: final decision
            
        )

    def forward(self, x):
        return self.network(x).squeeze()


# Initialise the model
model = CyberThreatDetector(input_dim=len(features))

# BCELoss measures how wrong our predictions are
# The closer to 0 the loss, the better the model
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Adam optimizer adjusts the model weights after each batch
# lr=0.001 is the learning rate — how big each adjustment is
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)


# ══════════════════════════════════════════════════════════════
# STEP 4: TRAIN THE MODEL
# Validation is used only to monitor performance while
# training. The test set remains untouched.
# ══════════════════════════════════════════════════════════════
best_f1 = 0.0

for epoch in range(12):

    # ---------------- TRAINING ----------------
    model.train()

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        preds = model(X_batch)

        loss = criterion(preds, y_batch)

        loss.backward()

        optimizer.step()

    # ---------------- VALIDATION ----------------
    model.eval()

    with torch.no_grad():

        val_logits = model(X_val_t)

        val_probs = torch.sigmoid(val_logits)

        threshold = 0.5   # .5 threshold remains the best

        val_preds_label = (val_probs >= threshold).float()

        y_true = y_val_t.cpu().numpy().astype(int).flatten()
        y_pred = val_preds_label.cpu().numpy().astype(int).flatten()

        val_accuracy = accuracy_score(y_true, y_pred)
        val_precision = precision_score(y_true, y_pred, zero_division=0)
        val_recall = recall_score(y_true, y_pred, zero_division=0)
        val_f1 = f1_score(y_true, y_pred, zero_division=0)

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "models/best_model.pth")
    print(
        f"Epoch {epoch+1:02d}/10 | "
        f"Acc={val_accuracy:.4f} | "
        f"Prec={val_precision:.4f} | "
        f"Recall={val_recall:.4f} | "
        f"F1={val_f1:.4f}"
    )
# ══════════════════════════════════════════════════════════════
# STEP 5: FINAL TEST EVALUATION
# The test set has never been used during training or model
# selection. It is evaluated exactly once to estimate how the
# model will perform on completely unseen data.
# ══════════════════════════════════════════════════════════════

model.load_state_dict(torch.load("models/best_model.pth"))
model.eval()

with torch.no_grad():

    # Get the raw outputs (logits)
    test_logits = model(X_test_t)

    # Convert logits to probabilities
    test_probs = torch.sigmoid(test_logits)

    # Classify as malicious if probability >= threshold{0.5}
    test_preds_label = (test_probs >= threshold).float()

    y_true = y_test_t.cpu().numpy().astype(int).flatten()
    y_pred = test_preds_label.cpu().numpy().astype(int).flatten()

    test_accuracy = accuracy_score(y_true, y_pred)
    test_precision = precision_score(y_true, y_pred, zero_division=0)
    test_recall = recall_score(y_true, y_pred, zero_division=0)
    test_f1 = f1_score(y_true, y_pred, zero_division=0)
    test_cm = confusion_matrix(y_true, y_pred)

print("\n════════ FINAL TEST PERFORMANCE ════════")

print(f"Accuracy : {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1 Score : {test_f1:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

# ══════════════════════════════════════════════════════════════
# STEP 6: SAVE THE TRAINED MODEL
# Save both the trained neural network and the fitted scaler.
# They can later be loaded to make predictions on new data
# without retraining the model.
# ══════════════════════════════════════════════════════════════

torch.save(model.state_dict(), "models/best_model.pth")

joblib.dump(scaler, "models/scaler.pkl")

print("\nBest model saved as 'models/best_model.pth'")
print("Scaler saved as 'models/scaler.pkl'")

#Accuracy measures overall correctness
#Precision measures how many predicted attacks were actually attack
#Recall measures how many real attacks were detected
#F1 Score balances Precision and Recall

Epoch 01/10 | Acc=0.7733 | Prec=0.5000 | Recall=0.0588 | F1=0.1053
Epoch 02/10 | Acc=0.7333 | Prec=0.2857 | Recall=0.1176 | F1=0.1667
Epoch 03/10 | Acc=0.7200 | Prec=0.2500 | Recall=0.1176 | F1=0.1600
Epoch 04/10 | Acc=0.6933 | Prec=0.2500 | Recall=0.1765 | F1=0.2069
Epoch 05/10 | Acc=0.6800 | Prec=0.2941 | Recall=0.2941 | F1=0.2941
Epoch 06/10 | Acc=0.6533 | Prec=0.2632 | Recall=0.2941 | F1=0.2778
Epoch 07/10 | Acc=0.6000 | Prec=0.2400 | Recall=0.3529 | F1=0.2857
Epoch 08/10 | Acc=0.5867 | Prec=0.2308 | Recall=0.3529 | F1=0.2791
Epoch 09/10 | Acc=0.5467 | Prec=0.2069 | Recall=0.3529 | F1=0.2609
Epoch 10/10 | Acc=0.5733 | Prec=0.2222 | Recall=0.3529 | F1=0.2727
Epoch 11/10 | Acc=0.5733 | Prec=0.2222 | Recall=0.3529 | F1=0.2727
Epoch 12/10 | Acc=0.5733 | Prec=0.2222 | Recall=0.3529 | F1=0.2727

════════ FINAL TEST PERFORMANCE ════════
Accuracy : 0.6800
Precision: 0.3000
Recall   : 0.3750
F1 Score : 0.3333

Confusion Matrix:
[[45 14]
 [10  6]]

Best model saved as 'models/best_model.pth'